# ant

> Read, write, and build Claude Code session transcripts

In [ ]:
#| default_exp ant

Claude Code stores every conversation as a JSONL transcript, and `claude --resume <session-id>` rebuilds a conversation from one. It does not care who wrote the file. A transcript assembled by hand, including tool calls that never really ran, resumes like any other. So sessions can be mined for data, saved as templates, or built synthetically to give a fresh session worked examples of tool use already in its context. This module finds, reads, and writes them.

In [ ]:
#| export
import base64, json, os, re, uuid
from datetime import datetime, timezone
from fastcore.utils import *
from fastllm.anthropic import denorm_msgs
from fastllm.chat import Msg, Part, PartType, mk_msg, hist2fmt, data_url
from llmsurgery.hist import dlg2canon
from llmsurgery.dialog import *

In [ ]:
from fastcore.test import *
from fastllm.chat import tool_dtls_tag, fmt2hist
from llmsurgery.dialog import *
from collections import Counter
import tempfile, shutil

## Where sessions live

Each project gets a folder under `~/.claude/projects`, named by the project's absolute path with every character that is not a letter or digit replaced by `-`. The path is resolved first, which matters on macOS, where `/tmp` and `/var` are symlinks into `/private`. The folder for a project in `/tmp/foo` is therefore `-private-tmp-foo`.

In [ ]:
#| export
SESSIONS = Path.home()/'.claude'/'projects'

def sess_dir(
    cwd=None, # Project directory; the current directory if None
):
    "The folder where Claude Code keeps session transcripts for the project at `cwd`"
    return SESSIONS/re.sub(r'[^a-zA-Z0-9]', '-', str(Path(cwd or '.').resolve()))

In [ ]:
sess_dir()

Path('/Users/jhoward/.claude/projects/-Users-jhoward-aai-ws-fastllm-claude-code-nbs')

Underscores are replaced too, which is easy to get wrong when sanitizing by hand:

In [ ]:
test_eq(sess_dir('/a/b_c').name, '-a-b-c')

Claude Code exports a session id as `CLAUDE_CODE_SESSION_ID` to every process it spawns, shell commands and MCP servers alike. But it identifies the current run, not the conversation: `claude --resume` and post-compaction restarts put a fresh id in the environment, while records keep appending to the original transcript, which is named by the conversation's first session id. So the env var names the transcript only while a conversation is on its first run, and can advertise an id that names no file at all. The reliable name for "this conversation" is instead the most recently modified transcript in the project's session folder: appends keep the live transcript's mtime freshest through restarts and resumes alike. `cur_sess` uses that heuristic, keeping the env var only as a fallback for when no transcript exists yet. One caveat: two conversations open on the same project trade the newest spot on every write, so under concurrent sessions pass an explicit `sid` (clikernel sidesteps this by resolving its host conversation once, at worker spawn, when the spawning conversation's transcript is freshest).

In [ ]:
#| export
def cur_sess(
    cwd=None, # Project directory; the current directory if None
):
    "The current conversation's session id: the most recent transcript for the project at `cwd`, else the advertised id"
    try: return max(sess_dir(cwd).glob('*.jsonl'), key=lambda p: p.stat().st_mtime).stem
    except ValueError: return os.environ.get('CLAUDE_CODE_SESSION_ID')

In [ ]:
cur_sess()

The newest transcript wins; with no transcripts at all, the advertised id is used:

In [ ]:
tp = Path(tempfile.mkdtemp())
sd = sess_dir(tp)
sd.mkdir(parents=True)
for i,n in enumerate(['older','newer']):
    (sd/f'{n}.jsonl').touch()
    os.utime(sd/f'{n}.jsonl', (i,i))
test_eq(cur_sess(tp), 'newer')
test_eq(cur_sess(tempfile.mkdtemp()), os.environ.get('CLAUDE_CODE_SESSION_ID'))

In [ ]:
#| export
def sess_file(
    sid=None, # Session id; `cur_sess(cwd)` if None
    cwd=None, # Project directory; the current directory, then all projects, if None
):
    "Path to the transcript of session `sid` for the project at `cwd`"
    sid = sid or cur_sess(cwd)
    p = sess_dir(cwd)/f'{sid}.jsonl'
    if cwd is not None or p.exists(): return p
    return first(SESSIONS.glob(f'*/{sid}.jsonl')) or p

Under Claude Code, `sess_file()` with no arguments is therefore the running conversation's own transcript, surviving resumes and restarts. Session ids are unique across projects, so when an explicit `sid`'s file is not in the current directory's folder, `sess_file` looks across all project folders, and the defaults work from anywhere, including a notebook kernel whose working directory is not the project root.

In [ ]:
test_eq(sess_file('abc', '/a/b_c'), SESSIONS/'-a-b-c'/'abc.jsonl')

## Writing a session

Records are plain dicts, so writing a session comes down to filling the envelope and linking the chain. `mk_rec` fills the envelope for one message. It writes the optional bookkeeping a real transcript carries (`version`, `gitBranch`, `userType`, `permissionMode`, and API metadata on assistant records), not only the six required fields: what Claude Code's LLM side makes of a sparse-but-valid record is close to untestable, so we err towards realistic.

Two records with the same content get different files by default, since ids and timestamps are fresh each call. Sometimes the opposite is wanted: the same history should produce byte-identical records, so the same session id maps to the same file however many times it is rebuilt. `canon` gives a canonical JSON rendering to hash, and `stable_uuid` turns any string into a deterministic uuid. `fastllm_claude_code.core` derives its session and record ids this way, and a session template built from a fixed script can too.

In [ ]:
#| export
CC_VERSION = '2.1.206'

def canon(o):
    "Canonical compact JSON for `o`, key-sorted, for stable hashing"
    return json.dumps(o, sort_keys=True, separators=(',', ':'), ensure_ascii=False)

def stable_uuid(s):
    "A uuid deterministically derived from string `s`"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, s))

In [ ]:
test_eq(canon(dict(b=1, a=2)), canon(dict(a=2, b=1)))
test_eq(stable_uuid('x'), stable_uuid('x'))
assert stable_uuid('x') != stable_uuid('y')

In [ ]:
#| export
def _now(): return datetime.now(timezone.utc).strftime(r'%Y-%m-%dT%H:%M:%S.%f')[:-3]+'Z'

def _txts(o, skip=()):
    if isinstance(o, str): yield o
    elif isinstance(o, dict): yield from (t for k,v in o.items() if k not in skip for t in _txts(v, skip))
    elif is_listy(o): yield from (t for x in o for t in _txts(x, skip))

def _est_toks(o):
    "Rule-of-thumb token estimate for every string in `o`: words * 1.5"
    return int(sum(len(s.split()) for s in _txts(o))*1.5)

def mk_rec(
    role, # 'user' or 'assistant'
    content, # A string, or a list of content blocks
    cwd='.', # Project directory recorded in the envelope
    uid=None, # Record uuid; random if None
    ts=None, # ISO8601 timestamp; the current time if None
    model='claude-sonnet-4-6', # Recorded in assistant API metadata; None omits it, so resume uses the user's default
    input_toks=0, # `input_tokens` recorded in assistant usage, e.g. an estimate of the context so far
    **kwargs, # Extra or overriding envelope fields, e.g. `isCompactSummary=True`
):
    "A transcript record for one conversation message, ready for `save_sess`"
    uid = uid or str(uuid.uuid4())
    if role=='user' and isinstance(content, list) and len(content)==1 and content[0].get('type')=='text': content = content[0]['text']
    msg = dict(type='message', role=role, content=content)
    r = dict(type=role, uuid=uid, parentUuid=None, sessionId=None, timestamp=ts or _now(), cwd=str(Path(cwd).resolve()),
        version=CC_VERSION, gitBranch='HEAD', isSidechain=False, userType='external', permissionMode='default', message=msg)
    if role=='assistant':
        tu = isinstance(content, list) and any(isinstance(b, dict) and b.get('type')=='tool_use' for b in content)
        r['requestId'] = 'req_'+stable_uuid(f'{uid}:req').replace('-', '')[:24]
        usage = dict(input_tokens=input_toks, output_tokens=_est_toks(content), cache_creation_input_tokens=0, cache_read_input_tokens=0)
        msg.update(id='msg_'+stable_uuid(f'{uid}:msg').replace('-', '')[:24],
            stop_reason='tool_use' if tu else 'end_turn', stop_sequence=None, stop_details=None, usage=usage)
        if model: msg['model'] = model
    return dict(r, **kwargs)

In [ ]:
mk_rec('user', 'Hello!')

{'type': 'user',
 'uuid': 'cefd6a28-ffad-42d3-aeff-53cb1563d942',
 'parentUuid': None,
 'sessionId': None,
 'timestamp': '2026-07-10T06:59:59.445Z',
 'cwd': '/Users/jhoward/aai-ws/fastllm-claude-code/nbs',
 'version': '2.1.206',
 'gitBranch': 'HEAD',
 'isSidechain': False,
 'userType': 'external',
 'permissionMode': 'default',
 'message': {'type': 'message', 'role': 'user', 'content': 'Hello!'}}

Assistant records get deterministic API metadata derived from the record id, and `stop_reason` reflects a trailing tool call:

In [ ]:
tu = [dict(type='tool_use', id='toolu_01', name='probe', input={})]
r = mk_rec('assistant', tu, uid=stable_uuid('demo'), ts='2026-01-01T00:00:00.000Z')
test_eq(r['message']['stop_reason'], 'tool_use')
test_eq(r, mk_rec('assistant', tu, uid=stable_uuid('demo'), ts='2026-01-01T00:00:00.000Z'))
assert 'requestId' in r and 'requestId' not in mk_rec('user', 'hi')
test_eq(r['message']['usage']['output_tokens'], _est_toks(tu))
assert 'model' not in mk_rec('assistant', tu, model=None)['message']
test_eq(mk_rec('user', [dict(type='text', text='hi')])['message']['content'], 'hi')  # lone text block: written as a string, as Claude Code does

`save_sess` assigns a session id, chains each record to the one before, and writes the file where `claude --resume` will look for it. It re-links `parentUuid` unconditionally, so it is for writing linear conversations. To copy a session while keeping its branch structure, write the records yourself.

In [ ]:
#| export
def save_sess(
    recs, # Records in conversation order, e.g. from `mk_rec`
    sid=None, # Session id; a fresh uuid if None
    cwd=None, # Project directory; the current directory if None
    ts=None, # If given, stamp every record's timestamp: True for the current time, or an ISO8601 string
):
    "Chain `recs`, write them as session `sid` for the project at `cwd`, and return `sid`"
    sid,prev = sid or str(uuid.uuid4()),None
    for r in recs:
        r['sessionId'],r['parentUuid'],prev = sid,prev,r['uuid']
        if ts: r['timestamp'] = _now() if ts is True else ts
        if 'session_id' in r: r['session_id'] = sid
    f = sess_file(sid, cwd or '.')
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(''.join(json.dumps(r)+'\n' for r in recs))
    return sid

Whole conversations convert in one call: `msgs2recs` takes Anthropic-style messages (dicts with `role` and `content`, like `claude_mk_msg` or fastllm's `denorm_msgs` produce) and builds one deterministic record per message, ready for `save_sess`. The same messages and key give the same ids, so a rebuilt session file is byte-identical.

In [ ]:
#| export
def msgs2recs(
    msgs, # Anthropic-style messages: dicts with `role` and `content`
    key='', # Salt: the same messages and key give the same ids
    cwd='.', # Project directory recorded in the envelopes
    ts='2026-01-01T00:00:00.000Z', # Timestamp for every record
    model='claude-sonnet-4-6', # Recorded in assistant API metadata; None omits it, so resume uses the user's default
    **kwargs, # Extra envelope fields for every record, e.g. `entrypoint`
):
    "Deterministic transcript records for `msgs`, one record per message"
    out,tot = [],0
    for i,m in enumerate(msgs):
        out.append(mk_rec(m['role'], m['content'], cwd=cwd, uid=stable_uuid(f'{key}:{i}:{canon(m)}'), ts=ts, model=model, input_toks=tot, **kwargs))
        tot += _est_toks(m['content'])
    return out

In [ ]:
den = [dict(role='user', content='Ping?'), dict(role='assistant', content=[dict(type='text', text='Pong.')])]
r1,r2 = msgs2recs(den, 'k'),msgs2recs(den, 'k')
test_eq(canon(r1), canon(r2))
test_ne(r1[0]['uuid'], msgs2recs(den, 'other')[0]['uuid'])
test_eq([r['type'] for r in r1], ['user','assistant'])
test_eq(r1[1]['message']['usage'], dict(input_tokens=1, output_tokens=3, cache_creation_input_tokens=0, cache_read_input_tokens=0))

## Synthetic tool calls

A worked tool call is two records joined by one id: a `tool_use` block in an assistant record, answered by a `tool_result` block in the user record that follows. `mk_tu` and `mk_tr` build the pair so the ids cannot drift, and `tool_turn` assembles the full exchange, from request to reply.

In [ ]:
#| export
def mk_tu(
    name, # Tool name, as the transcript records it
    input=None, # Tool arguments
    tid=None, # tool_use id; random if None
):
    "A `tool_use` content block"
    return dict(type='tool_use', id=tid or 'toolu_'+uuid.uuid4().hex[:24], name=name, input=input or {})

def mk_tr(
    tu, # The `tool_use` block being answered
    content, # The tool's output
):
    "The `tool_result` content block answering `tu`"
    return dict(type='tool_result', tool_use_id=tu['id'], content=content)

def tool_turn(
    prompt, # The user request
    name, # Tool name
    input, # Tool arguments
    output, # Tool result
    answer, # The assistant's closing text
    **kwargs, # Passed to each `mk_rec`, e.g. `cwd`
):
    "A complete synthetic tool-use turn, as four records ready for `save_sess`"
    tu = mk_tu(name, input)
    return [mk_rec('user', prompt, **kwargs), mk_rec('assistant', [tu], **kwargs),
        mk_rec('user', [mk_tr(tu, output)], **kwargs), mk_rec('assistant', [dict(type='text', text=answer)], **kwargs)]

The sample session in the next section is built from exactly one such turn.

## A sample session

The smallest useful synthetic history is a tool call that never ran, whose result carries a fact the model could not know any other way. We write it against a scratch project directory.

In [ ]:
proj = Path(tempfile.mkdtemp())
sample = tool_turn('Measure the flux please.', 'flux_meter', {}, 'flux: 41.7 kilofinches',
    'The flux reading is 41.7 kilofinches.', cwd=proj)
sid = save_sess(sample, cwd=proj)
sid

'f6d6657f-3c3b-46a0-a13a-1cc8915c2f73'

## Reading a session

A transcript is one JSON object per line. `load_sess` wraps each in `dict2obj` so fields read as attributes.

In [ ]:
#| export
def load_sess(
    sid=None, # Session id; the current session if None
    cwd=None, # Project directory; the current directory if None
):
    "The records of session `sid`, as an `L` of attribute-access dicts"
    return L(dict2obj(json.loads(l)) for l in sess_file(sid, cwd).read_text().splitlines())

Reading it back gives exactly what we wrote:

In [ ]:
back = load_sess(sid, proj)
test_eq(len(back), 4)
test_eq(back[-1].message.content[0].text, 'The flux reading is 41.7 kilofinches.')
test_eq(back[2].message.content[0].tool_use_id, back[1].message.content[0].id)

A record carries more than resume strictly needs. Only six fields are required: `type`, `uuid`, `parentUuid`, `sessionId`, `timestamp`, and `message`. The rest is optional bookkeeping. Strip `timestamp` and the session is not even found. `message` is shaped exactly as the Anthropic API shapes messages: a `role`, plus `content` as either a string or a list of content blocks (`text`, `tool_use`, `tool_result`, `thinking`). Assistant records in real transcripts also carry API metadata (`requestId`, `message.id`, `model`, usage), and none of it is needed on resume. In particular, synthetic histories work without `thinking` blocks.

## The parent chain

Resume does not replay the file top to bottom. Reconstruction starts at the last record and walks `parentUuid` links backwards, so a record nothing links to is dropped (the CLI prints a warning). Rewinding a conversation is what creates such records: the abandoned turns stay in the file, off the final chain. `sess_thread` performs the same walk.

In [ ]:
#| export
def sess_thread(
    recs, # Session records, e.g. from `load_sess`
):
    "The records on the active conversation chain, walking `parentUuid` back from the last record"
    byid = {r.uuid:r for r in recs if 'uuid' in r}
    cur,res = recs.filter(lambda r: 'uuid' in r)[-1],[]
    while cur is not None:
        res.append(cur)
        cur = byid.get(cur.get('parentUuid'))
    return L(reversed(res))

On the sample, every record is on the chain:

In [ ]:
test_eq(sess_thread(back).attrgot('uuid'), back.attrgot('uuid'))

Break a link and the walk stops early, mirroring what resume does with unchained records:

In [ ]:
broken = load_sess(sid, proj)
broken[2].parentUuid = None
test_eq(len(sess_thread(broken)), 2)

## Searching a session

A long conversation accumulates thousands of records, and `sess_thread` deliberately stops at the last compaction boundary, where the parent chain breaks. Finding where something was said or decided is a different job: search every record's text, then read the records around the hit.

In [ ]:
#| export
def rec_txt(
    r, # A session record
):
    "Every readable string in `r`'s message content, joined, for finding records by text"
    return '\n'.join(_txts(obj2dict(r).get('message', {}).get('content', ''), skip=('type','id','tool_use_id','signature')))

Content blocks nest (a `tool_result` can itself hold a list of text blocks), so `rec_txt` walks the whole message and joins every readable string it finds, skipping bookkeeping fields — block types, tool ids, and thinking signatures — that would otherwise pollute matches. On the sample, the planted fact appears in the tool result and in the reply that quotes it:

In [ ]:
test_eq([r['type'] for r in sample if 'kilofinches' in rec_txt(r)], ['user','assistant'])

In [ ]:
#| export
def conv_recs(
    recs, # Session records, e.g. from `load_sess`
):
    "Just the records carrying conversation messages, dropping Claude Code's bookkeeping"
    return L(r for r in recs if r.get('type') in ('user','assistant') and 'message' in r)

def rec_role(
    r, # A session record
):
    "The conversational role of `r`: a user record carrying tool results counts as `tool`"
    r = obj2dict(r)
    c = r['message']['content']
    if r['type']=='user' and isinstance(c, list) and any(b.get('type')=='tool_result' for b in c): return 'tool'
    return r['type']

`conv_recs` is the searchable subset of a transcript, and `rec_role` labels each record the way `recs2msgs` classifies them: the `user` type covers both real prompts and the tool results riding back, and telling them apart is the first thing every reader of a transcript needs. On the sample session:

In [ ]:
test_eq(len(conv_recs(sample)), 4)
test_eq([rec_role(r) for r in sample], ['user','assistant','tool','assistant'])

In [ ]:
#| export
def _preview(t, m, maxlen):
    h = maxlen//2
    s,e = max(0, m.start()-h), min(len(t), m.end()+h)
    return ('…' if s else '') + t[s:e].replace('\n',' ') + ('…' if e<len(t) else '')

class SessHits(list):
    "Search hits with a match-centered preview per line"
    def __repr__(self): return '\n'.join(f"{h.i:5} {h.role:9} {h.ts[:16]} {h.preview}" for h in self)

def sess_search(
    pat, # Regex to find in conversation text
    sid=None, # Session id; the current session if None
    cwd=None, # Project directory; the current directory if None
    maxlen=160, # Preview characters shown around a hit's first match
):
    "Search every conversation record of a session, returning `SessHits` with the records on `.recs`"
    recs = conv_recs(load_sess(sid, cwd))
    res = SessHits()
    for i,r in enumerate(recs):
        if m := re.search(pat, t:=rec_txt(r)): res.append(AttrDict(i=i, role=rec_role(r), ts=r['timestamp'], preview=_preview(t, m, maxlen), rec=r))
    res.recs = recs
    return res

The result displays one line per hit — index, role, timestamp, and the text around the first match — and keeps the searched records on `.recs`, so a hit's index leads straight to its neighbors. Searching the whole file rather than the thread is deliberate: pre-compaction history is exactly what archaeology is usually after. On the sample session the planted fact appears in the tool result and the reply that reports it:

In [ ]:
hits = sess_search('kilofinches', sid, proj)
test_eq([h.role for h in hits], ['tool','assistant'])
hits

In [ ]:
#| export
def show_recs(
    recs, # Session records, e.g. a slice of `SessHits.recs`
    mx=500, # Characters of text shown per record
):
    "A readable transcript of the conversation records in `recs`"
    def _s(r):
        t = rec_txt(r)
        if len(t)>mx: t = t[:mx]+f'…[+{len(t)-mx} chars]'
        return f"--- {rec_role(r)} {r['timestamp'][:19]} ---\n{t}"
    return PrettyString('\n'.join(_s(r) for r in conv_recs(recs)))

`show_recs` is the reading half of the workflow: search, take a slice of `.recs` around an interesting hit, and read it as a conversation. It is deliberately tolerant where `recs2msgs` is strict — any stretch of any transcript renders, with long records truncated:

In [ ]:
s = show_recs(hits.recs[hits[0].i-2:], mx=60)
assert s.count('---\n')==4 and 'flux: 41.7' in s
s

## Curating a captured session

Synthetic construction suits a short demo. For anything longer, capture beats writing: run a real session, then keep the span worth replaying. The search tools above find the span; curation then needs two operations: dropping what resume does not need, and re-deriving ids so one capture gives one file.

Thinking blocks are usually the bulk of a captured assistant turn, and resume rebuilds a conversation without them. Live transcripts write one content block per assistant record, so stripping thinking means dropping whole records; `save_sess` re-chains whatever survives.

In [ ]:
#| export
def strip_think(
    recs, # Session records
):
    "Drop records whose message content is only `thinking` blocks; resume does not need them"
    def _keep(r):
        c = r.get('message', {}).get('content', '')
        return isinstance(c, str) or not all(b.get('type')=='thinking' for b in c)
    return L(recs).filter(_keep)

In [ ]:
think = mk_rec('assistant', [dict(type='thinking', thinking='Quiet planning.', signature='')])
test_eq(strip_think([*sample, think]), sample)

After thinking, tool traffic is the bulk: file dumps in results, whole files in edit-tool inputs. `trunc_tools` caps every string inside `tool_use` inputs and `tool_result` content, leaving structure intact and marking each cut with the count of characters dropped. Records come back as fresh dicts, originals untouched.

In [ ]:
#| export
def _trunc_deep(o, mx):
    if isinstance(o, str): return o if len(o)<=mx else o[:mx]+f'…[+{len(o)-mx} chars]'
    if isinstance(o, dict): return {k:_trunc_deep(v, mx) for k,v in o.items()}
    if is_listy(o): return [_trunc_deep(x, mx) for x in o]
    return o

def trunc_tools(
    recs, # Session records
    mx=2000, # Maximum characters per string in tool inputs and results
):
    "Copies of `recs` with strings in tool_use inputs and tool_result content truncated to `mx` characters"
    recs = [obj2dict(r) for r in recs]
    for r in recs:
        c = r.get('message', {}).get('content', '')
        if isinstance(c, str): continue
        for b in c:
            if b.get('type')=='tool_use': b['input'] = _trunc_deep(b.get('input', {}), mx)
            elif b.get('type')=='tool_result' and 'content' in b: b['content'] = _trunc_deep(b['content'], mx)
    return L(recs)

A big tool result shrinks to the cap plus a marker; the small input alongside it, and the original records, are untouched:

In [ ]:
lt = tool_turn('Read it all.', 'reader', dict(path='/tmp/big.txt'), 'line\n'*500, 'Long.')
tr = trunc_tools(lt, 100)
assert rec_txt(tr[2]).endswith('…[+2400 chars]')
test_eq(tr[1]['message']['content'][0]['input'], dict(path='/tmp/big.txt'))
test_eq(lt[2]['message']['content'][0]['content'], 'line\n'*500)

Captured ids are random, so saving the same curated span twice gives two files that differ in every line. `reid_recs` re-derives them from position and a salt instead: record uuids, `tool_use`/`tool_result` pairs, assistant API metadata, and any envelope field that referenced a renamed uuid (such as `sourceToolAssistantUUID`). Records come back as fresh dicts; the originals are untouched.

In [ ]:
#| export
def reid_recs(
    recs, # Records in conversation order
    key='', # Salt: the same records and key give the same ids
    ts=None, # If given, set every record's timestamp to this
):
    "Deterministically re-derive record uuids, tool_use ids, and API metadata, so one capture gives one file"
    recs,ids = [obj2dict(r) for r in recs],{}
    for i,r in enumerate(recs):
        c = r.get('message', {}).get('content', '')
        if not isinstance(c, str):
            for j,b in enumerate(c):
                if b.get('type')=='tool_use':
                    nid = 'toolu_'+stable_uuid(f'{key}:{i}:{j}').replace('-', '')[:24]
                    ids[b['id']],b['id'] = nid,nid
                elif b.get('type')=='tool_result' and b.get('tool_use_id') in ids: b['tool_use_id'] = ids[b['tool_use_id']]
        uid = stable_uuid(f'{key}:{i}')
        if 'uuid' in r: ids[r['uuid']] = uid
        r['uuid'] = uid
        if ts: r['timestamp'] = ts
        if r.get('type')=='assistant':
            r['requestId'] = 'req_'+stable_uuid(f'{uid}:req').replace('-', '')[:24]
            r['message']['id'] = 'msg_'+stable_uuid(f'{uid}:msg').replace('-', '')[:24]
    for r in recs:
        for k,v in r.items():
            if isinstance(v, str) and v in ids: r[k] = ids[v]
    return L(recs)

Re-identified records keep the tool pairing, differ under a different salt, and leave the originals alone:

In [ ]:
r1,r2 = reid_recs(sample, 'tmpl'),reid_recs(sample, 'tmpl')
test_eq(canon(list(r1)), canon(list(r2)))
test_eq(r1[2]['message']['content'][0]['tool_use_id'], r1[1]['message']['content'][0]['id'])
test_ne(r1[0]['uuid'], reid_recs(sample, 'other')[0]['uuid'])
test_ne(r1[1]['message']['content'][0]['id'], sample[1]['message']['content'][0]['id'])

With the timestamps pinned too, the whole file is byte-for-byte reproducible. A captured record can also carry a `session_id` field alongside `sessionId`; `save_sess` keeps both in step:

In [ ]:
t1,t2 = reid_recs(sample, 'tmpl', ts='2026-01-01T00:00:00.000Z'),reid_recs(sample, 'tmpl', ts='2026-01-01T00:00:00.000Z')
t1[0]['session_id'] = t2[0]['session_id'] = 'stale'
tid = save_sess(t1, stable_uuid('tmpl'), proj)
b = sess_file(tid, proj).read_bytes()
test_eq(save_sess(t2, stable_uuid('tmpl'), proj), tid)
test_eq(sess_file(tid, proj).read_bytes(), b)
test_eq(load_sess(tid, proj)[0].session_id, tid)

`fork_sess` puts the whole diet on one line: load a session, optionally drop thinking and truncate tool traffic, and write the result under a fresh id, leaving the original untouched. Resume the returned id to compare the munged conversation against the original. With a `key`, ids re-derive deterministically, so re-running the same munge overwrites its fork instead of accumulating copies.

In [ ]:
#| export
def fork_sess(
    sid=None, # Session id to fork; `cur_sess()` if None
    cwd=None, # Project directory; passed to `sess_file` via `load_sess`
    mx=None, # If given, truncate tool input/output strings to `mx` characters
    think=True, # Keep thinking records?
    key=None, # If given, record and session ids re-derive deterministically from this salt
):
    "Write a munged copy of session `sid` under a fresh id, returning the new id to resume"
    recs = sess_thread(load_sess(sid, cwd))
    if not think: recs = strip_think(recs)
    if mx: recs = trunc_tools(recs, mx)
    if key: recs = reid_recs(recs, key)
    return save_sess(list(recs), stable_uuid(key) if key else None, cwd)

Forking the sample with a thinking record appended and a tight cap: the fork loses the thinking record and truncates the tool result, the original keeps all five records, and the same call gives the same fork id:

In [ ]:
tsid = save_sess(reid_recs([*sample, think], 'munge'), stable_uuid('munge'), proj)
fkid = fork_sess(tsid, proj, mx=10, think=False, key='fork1')
test_eq(len(load_sess(fkid, proj)), 4)
assert rec_txt(load_sess(fkid, proj)[2]).endswith('…[+12 chars]')
test_eq(len(load_sess(tsid, proj)), 5)
test_eq(fork_sess(tsid, proj, mx=10, think=False, key='fork1'), fkid)

## From dialogs

An authored dialog converts to a session in two steps: `dlg2canon` (in `hist`) recovers the canonical messages with real tool calls, and `denorm_msgs` from `fastllm.anthropic` shapes them for Claude. `dlg2sess` then writes the records. The session id derives from the dialog name, so re-converting an edited dialog overwrites its session rather than accumulating copies.

In [ ]:
#| export
def dlg2msgs(
    dlg, # A `Dialog`, ending with a prompt
    aim_info=None, # Model capability dict for media handling; images enabled if None
):
    "Anthropic-style messages for `dlg`, with each reply's tool calls recovered as real blocks"
    return denorm_msgs(dlg2canon(dlg, aim_info))

def dlg2sess(
    dlg, # The dialog to convert
    cwd=None, # Project directory for the session; the current directory if None
    key='dlg2sess', # Salt for deterministic record ids
    aim_info=None, # Model capability dict; images enabled if None
):
    "Write `dlg` as a Claude Code session for the project at `cwd`, returning the session id to resume"
    recs = msgs2recs(dlg2msgs(dlg, aim_info), key=key, cwd=str(Path(cwd or '.').resolve()))
    return save_sess(recs, stable_uuid(f'{key}:{dlg.name}'), cwd)

A dialog whose reply used a tool, with a referenced image attachment, exercises the whole path. The tool call comes back as paired `tool_use`/`tool_result` blocks, and the attachment becomes a base64 image block, both exactly as real transcripts carry them:

In [ ]:
def tool_dtl(func, args, result):
    "A tool-call details block in the reply format `fmt2hist` parses"
    d = json.dumps({'id':'call1', 'server':False, 'call':{'function':func, 'arguments':args}, 'result':result})
    return f"{tool_dtls_tag}\n<summary><code>{func}(...)</code></summary>\n\n```json\n{d}\n```\n\n</details>"

png = tiny_png
fdlg = Dialog('flux')
fatt = Attachment(png, 'image/png')
fdlg.mk_message(f'The rig: ![](attachment:{fatt.id})', msg_type=snote, attachments=[fatt])
freply = f"Let me check.\n\n{tool_dtl('flux_meter', {'unit':'kf'}, 'flux: 41.7 kilofinches')}\n\nThe flux is 41.7 kilofinches."
fdlg.mk_message('Measure the flux please.', msg_type=sprompt, output=freply)
fmsgs = dlg2msgs(fdlg)
[(m['role'], [b['type'] for b in m['content']] if isinstance(m['content'], list) else 'str') for m in fmsgs]

In [ ]:
test_eq([m['role'] for m in fmsgs], ['user','assistant','user','assistant'])
ftu = first(b for b in fmsgs[1]['content'] if b['type']=='tool_use')
ftr = first(b for b in fmsgs[2]['content'] if b['type']=='tool_result')
test_eq(ftr['tool_use_id'], ftu['id'])
fimg = first(b for b in fmsgs[0]['content'] if b['type']=='image')
test_eq(fimg['source']['media_type'], 'image/png')

Roundtrip and determinism, into the same scratch project as the sample session:

In [ ]:
fsid = dlg2sess(fdlg, proj)
fback = load_sess(fsid, proj)
test_eq(len(fback), 4)
test_eq(sess_thread(fback).attrgot('uuid'), fback.attrgot('uuid'))
test_eq(dlg2sess(fdlg, proj), fsid)

And the live proof, resuming the dialog-built session and asking about the planted fact (spends tokens, so out of CI):

In [ ]:
#| eval: false
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
opts = ClaudeAgentOptions(resume=fsid, cwd=str(proj), model='haiku')
async for m in query(prompt='What did the flux_meter tool report, exactly?', options=opts):
    if isinstance(m, ResultMessage): print(m.result)

## Back to dialogs

The reverse direction turns a recorded session into a dialog, ready for editing before it becomes a template. `recs2msgs` normalizes conversation records to canonical fastllm messages: one `Msg` per record, with tool calls and results carrying the same part data that `fmt2hist` produces, and `tool_result` records taking the `tool` role. Thinking blocks are kept as `thinking` parts (run `strip_think` first to drop them), and any block type we cannot faithfully convert raises rather than silently degrading.

In [ ]:
#| export
def _tr_txt(b):
    c = b.get('content', '')
    if isinstance(c, str): return c
    if any(x.get('type') not in ('text','tool_reference') for x in c): raise ValueError('unsupported tool_result block')
    return '\n'.join(x['text'] if x['type']=='text' else f"<tool_reference tool=\"{x['tool_name']}\"/>" for x in c)

def _norm_user(content):
    if isinstance(content, str): return mk_msg(content)
    items = []
    for b in content:
        if b['type']=='text': items.append(b['text'])
        elif b['type']=='image': items.append(base64.b64decode(b['source']['data']))
        else: raise ValueError(f"unsupported user block: {b['type']}")
    return mk_msg(items)

def recs2msgs(
    recs, # Session records, e.g. from `load_sess`
):
    "Canonical fastllm messages for the conversation records in `recs`"
    msgs,names = [],{}
    for r in recs:
        r = obj2dict(r)
        if r.get('type') not in ('user','assistant'): continue
        c = r['message']['content']
        if isinstance(c, str) and r['type']=='assistant': c = [dict(type='text', text=c)]
        if r['type']=='assistant':
            parts = []
            for b in c:
                if b['type']=='text': parts.append(Part(type=PartType.text, text=b['text']))
                elif b['type']=='thinking': parts.append(Part(type=PartType.thinking, text=b.get('thinking','')))
                elif b['type']=='tool_use':
                    names[b['id']] = b['name']
                    parts.append(Part(type=PartType.tool_use, data=dict(id=b['id'], name=b['name'], arguments=b.get('input', {}))))
                else: raise ValueError(f"unsupported assistant block: {b['type']}")
            msgs.append(Msg(role='assistant', content=parts))
        elif rec_role(r)=='tool':
            if not all(b.get('type')=='tool_result' for b in c): raise ValueError('record mixes tool_result with other blocks')
            msgs.append(Msg(role='tool', content=[Part(type=PartType.tool_result, text=_tr_txt(b),
                data=dict(id=b['tool_use_id'], name=names.get(b['tool_use_id']))) for b in c]))
        else: msgs.append(_norm_user(c))
    return msgs

On the sample session, the four records become user, assistant, tool, and assistant messages, with the tool call and its result joined by one id:

In [ ]:
smsgs = recs2msgs(sample)
test_eq([m.role for m in smsgs], ['user','assistant','tool','assistant'])
test_eq(smsgs[0].text, 'Measure the flux please.')
test_eq(smsgs[1].content[0].data['id'], smsgs[2].content[0].data['id'])
test_eq(_tr_txt(dict(content=[dict(type='tool_reference', tool_name='probe')])), '<tool_reference tool="probe"/>')
test_eq(recs2msgs([mk_rec('assistant', 'hi')])[0].text, 'hi')

In [ ]:
#| export
def msgs2dlg(
    msgs, # Canonical messages, e.g. from `recs2msgs`
    name, # Dialog name
    cls=Dialog, # Dialog class to create
    mx=2000, # Maximum characters per rendered tool input/output string (see `hist2fmt`)
):
    "A dialog for `msgs`: one prompt per user turn, replies rendered in the format `fmt2hist` parses"
    dlg,turns = cls(name),[]
    for m in msgs:
        if m.role=='user': turns.append((m,[]))
        elif turns: turns[-1][1].append(m)
        else: raise ValueError('msgs must start with a user message')
    for u,rs in turns:
        segs,atts = [],[]
        for p in u.content:
            if p.type==PartType.text: segs.append(p.text)
            elif p.type==PartType.input_image:
                mime,data = data_url(p.text)
                atts.append(Attachment(base64.b64decode(data), mime))
                segs.append(f'![](attachment:{atts[-1].id})')
            else: raise ValueError(f'unsupported user part: {p.type}')
        dlg.mk_message('\n\n'.join(segs), msg_type=sprompt, output=hist2fmt(rs, mx=mx), attachments=atts)
    return dlg

Prompt content is the user turn verbatim: text parts joined by blank lines, and images pulled out as attachments with a reference left in the text. A session that was itself built from a dialog therefore converts back with its rendered XML wrapping (`<prompt>`, `<message type="note">`) still in place, not the original notes; the reverse direction is for sessions recorded live, where user content is plain.

In [ ]:
mdlg = msgs2dlg(smsgs, 'flux back')
test_eq(len(mdlg.messages), 1)
test_eq(mdlg.messages[0].content, 'Measure the flux please.')

The reply renders as text plus a tool-call details block, and `fmt2hist` parses it back to exactly the canonical messages it came from. This is the law that makes the conversion safe to edit: the dialog holds nothing the session format cannot round-trip.

In [ ]:
test_eq(fmt2hist(mdlg.messages[0].ai_output), smsgs[1:])

A pasted image survives the full circle: it becomes an attachment on the prompt, and converting the dialog onwards to messages yields the same base64 block the record held.

In [ ]:
ib = [dict(type='text', text='The rig:'), dict(type='image', source=dict(type='base64', media_type='image/png', data=base64.b64encode(tiny_png).decode()))]
irecs = msgs2recs([dict(role='user', content=ib), dict(role='assistant', content=[dict(type='text', text='Nice rig.')])])
idlg = msgs2dlg(recs2msgs(irecs), 'rig')
test_eq(idlg.messages[0].attachments[0].data, tiny_png)
assert f'attachment:{idlg.messages[0].attachments[0].id}' in idlg.messages[0].content
test_eq(first(b for b in dlg2msgs(idlg)[0]['content'] if b['type']=='image')['source']['data'], base64.b64encode(tiny_png).decode())

In [ ]:
#| export
def sess2dlg(
    sid=None, # Session id; `cur_sess()` if None
    cwd=None, # Project directory; passed to `sess_file` via `load_sess`
    name=None, # Dialog name; the session id if None
    mx=2000, # Maximum characters per rendered tool input/output string (see `hist2fmt`)
):
    "The conversation of session `sid` as a dialog, one prompt per user turn"
    recs = strip_think(sess_thread(load_sess(sid, cwd)))
    return msgs2dlg(recs2msgs(recs), name or sid or cur_sess(), mx=mx)

`sess2dlg` composes the whole read path: load the transcript, walk the parent chain, drop thinking records, and convert. Reading the sample session back gives the same dialog we built from its messages by hand:

In [ ]:
rdlg = sess2dlg(sid, proj, 'flux back')
test_eq(rdlg.messages[0].content, mdlg.messages[0].content)
test_eq(rdlg.messages[0].ai_output, mdlg.messages[0].ai_output)

## Inside a live session

Since Claude Code exports `CLAUDE_CODE_SESSION_ID` to child processes, code running inside a session can read the very transcript it is part of. The cells below do that when a live transcript exists, and fall back to the sample session otherwise (a plain shell, CI, or a resumed session whose advertised id names no file).

In [ ]:
recs = load_sess() if sess_file().exists() else load_sess(sid, proj)
len(recs)

4

The conversation itself is the `user` and `assistant` records. The rest is bookkeeping Claude Code adds as it runs: `attachment` for injected context such as skills and file contents, `system` for hook output, and assorted prompt, mode, and snapshot markers. Compaction adds a `user` record flagged `isCompactSummary`, carrying a summary of everything before it.

In [ ]:
Counter(recs.attrgot('type'))

Counter({'user': 2, 'assistant': 2})

Here is one user record in full:

In [ ]:
first(recs, lambda r: r.type=='user' and isinstance(r.get('message',{}).get('content'), str))

<div class="prose" markdown="1">

```python
{ 'cwd': '/private/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/tmpwm8r1gdg',
  'gitBranch': 'HEAD',
  'isSidechain': False,
  'message': { 'content': 'Measure the flux please.',
               'role': 'user',
               'type': 'message'},
  'parentUuid': None,
  'permissionMode': 'default',
  'sessionId': 'f6d6657f-3c3b-46a0-a13a-1cc8915c2f73',
  'timestamp': '2026-07-10T06:59:59.447Z',
  'type': 'user',
  'userType': 'external',
  'uuid': '754795a5-469f-44ba-8e61-ab31112b99a1',
  'version': '2.1.206'}
```

</div>

In [ ]:
t = sess_thread(recs)
len(t), len(recs)

(4, 4)

In a live transcript the gap between the two counts is bookkeeping records that carry no `uuid`, plus any abandoned branches; the sample has neither. Consecutive records on the chain link up:

In [ ]:
assert all(b.parentUuid==a.uuid for a,b in zip(t, t[1:]))

The proof that the sample is a working session: resume it and ask about the planted fact. The Claude Agent SDK drives the same CLI, so `resume` there reads the same files. This spends tokens, so it is excluded from automated tests.

In [ ]:
#| eval: false
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
opts = ClaudeAgentOptions(resume=sid, cwd=str(proj), model='haiku')
async for m in query(prompt='What is the flux reading? Reply with only the value.', options=opts):
    if isinstance(m, ResultMessage): print(m.result)

## Cleanup

Remove the sample from `~/.claude/projects`, along with the scratch project.

In [ ]:
shutil.rmtree(sess_dir(proj))
shutil.rmtree(proj)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()